# Data Proprocessing

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.preprocessing import OrdinalEncoder

import os, sys
# Add the 'scripts' directory to the Python path for module imports
sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))
# Set max rows and columns to display
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [ ]:
# Read the dataset
df = pd.read_csv('../data/cleaned_data.csv', low_memory=False, index_col=False)

In [ ]:
# Explore the first five rows
df.head(5)

In [ ]:
# Convert 'CapitalOutstanding' to numeric, coerce errors to handle non-numeric values (e.g., empty strings or NaNs)
df['CapitalOutstanding'] = pd.to_numeric(df['CapitalOutstanding'], errors='coerce')

# Fill missing values if necessary, e.g., with 0 or the column's mean
df['CapitalOutstanding'] = df['CapitalOutstanding'].fillna(0)  # Or df['CapitalOutstanding'].fillna(df['CapitalOutstanding'].mean())

# Ensure it's a float type
df['CapitalOutstanding'] = df['CapitalOutstanding'].astype(float)

# Verify the conversion
print(df['CapitalOutstanding'].dtype)

In [ ]:
# Get categorical columns
categorical_columns = df.select_dtypes(include='object').columns
# Create a summary DataFrame
summary_df = pd.DataFrame({
    'Column': categorical_columns,
    'DataType': [df[col].dtype for col in categorical_columns],
    'NumUniqueValues': [df[col].nunique() for col in categorical_columns]
})

# Print the summary DataFrame
summary_df

Key Insights from the Summary:

Columns like Language, Country, ItemType, StatutoryClass, and StatutoryRiskType don't provide variability.

Drop these columns from the dataset.

Model (411 unique values) could add complexity.

Consider reducing cardinality by grouping or applying target encoding.

Columns like AlarmImmobiliser, TrackingDevice, and NewVehicle have only 2 unique values.

TransactionMonth should be converted to a date format.

Extract useful features like year, month, and quarter if necessary.

CapitalOutstanding should be numeric and can be used as a continuous feature.

In [ ]:
columns_to_drop = ['Language', 'Country', 'ItemType', 'StatutoryClass', 'StatutoryRiskType']
df = df.drop(columns=columns_to_drop)

Convert the CapitalOutstanding to numeric format

In [ ]:
df.columns

In [ ]:
# Convert boolean 'IsVATRegistered' to integer
df['IsVATRegistered'] = df['IsVATRegistered'].astype(int)

### Convert the TransactionMonth to a datetime format and extract relevant features.

In [ ]:
df.columns

In [ ]:
df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'])
df['TransactionYear'] = df['TransactionMonth'].dt.year
df['TransactionMonthOnly'] = df['TransactionMonth'].dt.month
df['TransactionQuarter'] = df['TransactionMonth'].dt.quarter
# Drop the original datetime column if it's no longer needed
df = df.drop(columns=['TransactionMonth'])

Check the Distribution of Title, MaritalStatus, and Gender

In [ ]:
print(f'Gender Distribution:\n{df.Gender.value_counts()}')
print(f'Title Distribution:\n {df.Title.value_counts()}')
print(f'Marital Status Distribution:\n {df.MaritalStatus.value_counts()}')

In [ ]:
print(f'Gender Distribution:\n{df.Gender.value_counts()}')
print(f'Title Distribution:\n {df.Title.value_counts()}')
print(f'Marital Status Distribution:\n {df.MaritalStatus.value_counts()}')

High Proportion of Unspecified Gender:

A significant portion of the Gender values (950,526) is missing or unspecified.

These unspecified gender values can be filled using the information from the Title column where available.
Title-to-Gender Mapping:

Titles like Mr, Mrs, Miss, and Ms provide a clear indication of gender.
Dr is ambiguous and may not specify gender directly.
Approach to Impute Missing Genders Using Titles:

Mr ? Male

Mrs, Miss, Ms ? Female

Dr ? Leave as Not specified (since it?s ambiguous)

Balanced Gender Distribution:

The number of male and female entries is now significantly more balanced compared to the initial state where a large portion of the Gender values were unspecified.

Dropping the MaritalStatus column is a reasonable decision given the high proportion of unspecified values

In [ ]:
# Drop the 'MaritalStatus' column from the DataFrame
df = df.drop(columns=['MaritalStatus'])

In [ ]:
# Drop the specified columns from the DataFrame
df = df.drop(columns=['Unnamed: 0', 'UnderwrittenCoverID', 'PolicyID'])

Encoding Strategy:

Label Encoding: For binary or low-cardinality features.
One-Hot Encoding: For moderate cardinality features where the number of new columns remains manageable.
Target Encoding: For high cardinality features to reduce the number of new features while capturing useful patterns.
Frequency Encoding: Another option for high cardinality features, where you replace categories with their frequency in the dataset.

In [ ]:
# Define cardinality thresholds
low_cardinality_threshold = 10
high_cardinality_threshold = 20

# Identify categorical columns
categorical_cols = df.select_dtypes(include='object').columns

# Separate features by cardinality
low_cardinality_features = [col for col in categorical_cols if df[col].nunique() <= low_cardinality_threshold]
moderate_cardinality_features = [col for col in categorical_cols if low_cardinality_threshold < df[col].nunique() <= high_cardinality_threshold]
high_cardinality_features = [col for col in categorical_cols if df[col].nunique() > high_cardinality_threshold]

# Label Encoding for low cardinality features
label_encoders = {}
for feature in low_cardinality_features:
    le = LabelEncoder()
    df[feature] = le.fit_transform(df[feature])
    label_encoders[feature] = le

# One-Hot Encoding for moderate cardinality features
one_hot_encoder = OneHotEncoder(drop='first', sparse_output=False)
for feature in moderate_cardinality_features:
    encoded = one_hot_encoder.fit_transform(df[[feature]])
    encoded_df = pd.DataFrame(encoded, columns=one_hot_encoder.get_feature_names_out([feature]))
    df = pd.concat([df, encoded_df], axis=1)
    df = df.drop(columns=[feature])

# Frequency Encoding for high cardinality features (Example)
for feature in high_cardinality_features:
    freq = df[feature].value_counts()
    df[feature + '_freq'] = df[feature].map(freq)
    df = df.drop(columns=[feature])



# Save or proceed with further analysis/modeling
# df.to_csv('preprocessed_data.csv', index=False)  # Uncomment to save the preprocessed data

In [ ]:
# Verify the transformations
df.head()

Create new feature from TotalPremium and TotalClaims

In [ ]:
df['Premium_to_Claims_Ratio'] = df['TotalPremium'] / (df['TotalClaims'] + 1)  # +1 to avoid division by zero

Scaling Before Feature Selection :

Feature scaling should typically be performed before feature selection. This ensures that all features are on a comparable scale before we apply selection techniques like SelectKBest, which may rely on statistical methods sensitive to scale, such as f_classif or mutual_info_classif.

If features have different ranges, features with larger scales might dominate the selection process, making the selection biased toward those features. By scaling before selection, each feature contributes fairly to the selection criteria.

In [ ]:
# Check missing values in each column
missing_counts = df.isnull().sum()

# Print columns with more than 1 missing value
columns_with_missing = missing_counts[missing_counts > 1].index
print(f'Columns with more than 1 missing value:\n{columns_with_missing}')

# Drop these columns
df = df.drop(columns=columns_with_missing)

# Verify the result
print(f'Columns remaining after dropping:\n{df.columns}')

In [ ]:
# Separate Target and Feature Variables
X = df.drop(columns=['TotalPremium', 'TotalClaims'])
y = df['TotalPremium']  # or 'TotalClaims' depending on the target

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Feature Selection and Dimensionality Reduction

In [ ]:
#  Feature Selection using SelectKBest

selector = SelectKBest(score_func=f_classif, k=20)  # Select top 20 features
X_new = selector.fit_transform(X_scaled, y)

Reduce Dimensionality In large datasets with many features, some features might be redundant or highly correlated. PCA transforms these features into a smaller set of uncorrelated variables called principal components, which capture most of the variability in the data. This can lead to simpler, faster models that are less prone to overfitting.

In [ ]:
# Example: Dimensionality Reduction using PCA
pca = PCA(n_components=20)  # Reduce to 20 principal components
X_pca = pca.fit_transform(X)

In [ ]:
# Plot explained variance
plt.figure(figsize=(10, 4))
sns.lineplot(x=range(1, 21), y=np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Explained Variance')
plt.show()

In [ ]:
# Get the names of the selected features
selected_features = X.columns[selector.get_support()]
# Create a dataframe with the selected features and their corresponding PCA Componets
#df_result = pd.DataFrame(X_pca, columns=[f'PCA_{i+1}' for i in range(20)])
# df_result['Selected_Features'] = selected_features

In [ ]:
# Create new dataframe with selected features
df_new = pd.DataFrame(X_new, columns=selected_features)
# merge the target variable to the new dataframe
df_new['TotalPremium'] = y

In [ ]:
# Save the new dataframe to csv for model training
df_new.to_csv('../data/preprocessed_data.csv', index=False)

In [ ]:
df_new.head()

In [ ]:
df_new.mean()

In [ ]:
df_new.std()

In [ ]:
df.describe()